# Full PP-OCRv5 + KDL DocBench experiment on Colab L4

Notebook nay chay tron pipeline moi tren mot runtime Colab L4:

`pdf-inspector -> PP-OCRv5 -> BM25 top-20 -> KDL + pdf-inspector -> fixed chunking -> text-embedding-3-small -> baseline_legacy -> QA + judge`

PP-OCRv5 va KDL dung chung GPU, vi vay phai chay PP-OCRv5 xong, luu artifact, giai phong GPU roi moi khoi dong vLLM. Ket qua va checkpoint duoc ghi vao Google Drive de co the resume khi Colab bi ngat.

Truoc khi chay:

1. Chon runtime GPU **L4**.
2. Sua duong dan `DRIVE_DOCBENCH_ROOT` o cell cau hinh.
3. Dat `OPENROUTER_API_KEY` trong Colab Secrets. `HF_TOKEN` va `VLLM_API_KEY` la tuy chon.
4. Dam bao branch `colab-host` tren GitHub da co cac thay doi PP-OCRv5 trong repository.


In [ ]:
from google.colab import drive, userdata

drive.mount('/content/drive')

import gc
import hashlib
import json
import os
import re
import shlex
import shutil
import signal
import statistics
import subprocess
import sys
import time
import zipfile
from pathlib import Path

print('Python:', sys.version.split()[0])


In [ ]:
# ===================== EDIT THESE VALUES =====================
REPO_URL = 'https://github.com/iSE-UET-VNU/AXIOM_DE-RD.git'
GIT_BRANCH = 'colab-host'
REPO_DIR = Path('/content/AXIOM_DE-RD')

# Vi du: copy DocBench vao MyDrive/AXIOM_DE-RD/data/raw/...
DRIVE_DOCBENCH_ROOT = Path(
    '/content/drive/MyDrive/0.1. BENCHMARK'
)
# Upload mapping vao MyDrive/vidore_metadata/vidore_v3_{physics,industrial}/corpus/.
DRIVE_VIDORE_ROOT = Path('/content/drive/MyDrive/Colab Result/vidore_metadata')
DRIVE_RESULTS_ROOT = Path(
    '/content/drive/MyDrive/Colab Result'
)
RUN_NAME = 'on_demand_basic_ppocrv5_l4'
RUN_OUTPUT_DIR = DRIVE_RESULTS_ROOT / RUN_NAME
PP_OCR_WORK_DIR = RUN_OUTPUT_DIR / 'ppocrv5'

CONFIG_REL = Path('configs/pipeline.docbench-on-demand-basic.yaml')
RETRIEVAL_SCOPE = 'lake'
TOP_K_PAGES = 20
MAX_CONTEXT_CHUNKS = 3

# None = toan bo benchmark. Dat so nho de smoke test.
MAX_DOCUMENTS = None
QUESTION_LIMIT = None
SKIP_QA = True
SKIP_JUDGE = False
FORCE_REPARSE = False
FORCE_REBUILD_INDEX = False

# True chi khi dataset/tham so chon trang da thay doi.
REBUILD_LIGHT_BUNDLE = False
RERUN_PP_OCR = False
STOP_VLLM_AT_END = False

def get_secret(name: str) -> str:
    try:
        return str(userdata.get(name) or '')
    except Exception:
        return ''

OPENROUTER_API_KEY = get_secret('OPENROUTER_API_KEY')
VLLM_API_KEY = get_secret('VLLM_API_KEY')
HF_TOKEN = get_secret('HF_TOKEN')

if not OPENROUTER_API_KEY:
    raise RuntimeError(
        'Can OPENROUTER_API_KEY trong Colab Secrets de chay embedding, QA va judge.'
    )

RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PP_OCR_WORK_DIR.mkdir(parents=True, exist_ok=True)
PP_OCR_BUNDLE = PP_OCR_WORK_DIR / 'light_ocr_bundle.zip'
PP_OCR_JSONL = PP_OCR_WORK_DIR / 'light_ocr_ppocrv5.jsonl'
PP_OCR_RESULTS_ZIP = PP_OCR_WORK_DIR / 'light_ocr_results.zip'

print('Dataset:', DRIVE_DOCBENCH_ROOT)
print('Output:', RUN_OUTPUT_DIR)
print('Branch:', GIT_BRANCH)
print('OpenRouter key:', 'da nap' if OPENROUTER_API_KEY else 'chua nap')


In [ ]:
# Kiem tra GPU truoc khi cai PaddlePaddle.
subprocess.run(['nvidia-smi'], check=True)
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Khong tim thay CUDA. Hay chon runtime GPU L4.')
gpu = torch.cuda.get_device_properties(0)
vram_gib = gpu.total_memory / 1024**3
print(f'GPU: {gpu.name}')
print(f'VRAM: {vram_gib:.1f} GiB')
print(f'CUDA: {torch.version.cuda}')
print(f'CPU: {os.cpu_count() or 1}')
if 'L4' not in gpu.name:
    raise RuntimeError(f'Notebook nay duoc canh chinh cho L4, dang thay {gpu.name}.')
if vram_gib < 20:
    raise RuntimeError('GPU khong co du VRAM cho KDL nano; can L4 24 GiB tro len.')
del torch


In [ ]:
# Clone/pull dung branch. Neu repo da co thay doi local thi giu nguyen de tranh mat du lieu.
if REPO_DIR.exists() and not (REPO_DIR / '.git').is_dir():
    raise RuntimeError(f'{REPO_DIR} ton tai nhung khong phai Git repository.')

if not REPO_DIR.exists():
    subprocess.run(
        ['git', 'clone', '--single-branch', '--branch', GIT_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    dirty = subprocess.check_output(
        ['git', 'status', '--porcelain'], cwd=REPO_DIR, text=True
    ).strip()
    if dirty:
        print('Repo co thay doi local; bo qua pull va dung source hien tai.')
    else:
        subprocess.run(['git', 'fetch', 'origin', GIT_BRANCH], cwd=REPO_DIR, check=True)
        current = subprocess.check_output(
            ['git', 'branch', '--show-current'], cwd=REPO_DIR, text=True
        ).strip()
        if current != GIT_BRANCH:
            subprocess.run(['git', 'checkout', GIT_BRANCH], cwd=REPO_DIR, check=True)
        subprocess.run(['git', 'pull', '--ff-only', 'origin', GIT_BRANCH], cwd=REPO_DIR, check=True)

required = [
    REPO_DIR / 'scripts' / 'build_ppocrv5_bundle.py',
    REPO_DIR / 'research' / 'data_discovery' / 'run_docbench_e2e.py',
    REPO_DIR / CONFIG_REL,
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    available_configs = sorted(str(path.relative_to(REPO_DIR)) for path in (REPO_DIR / 'configs').glob('*.yaml'))
    raise RuntimeError(
        'Thieu file can thiet trong repo: ' + ', '.join(missing) + '\n'
        + 'Config hien co: ' + ', '.join(available_configs)
    )

commit = subprocess.check_output(
    ['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO_DIR, text=True
).strip()
sys.path.insert(0, str(REPO_DIR))
os.environ['PYTHONPATH'] = f'{REPO_DIR}:{os.environ.get("PYTHONPATH", "")}'
print('Using commit:', commit)


In [ ]:
# Cai dependency cua repository truoc PP-OCRv5; chua cai vLLM de tranh tranh chap GPU libs.
def pip_install(*args: str) -> None:
    command = [sys.executable, '-m', 'pip', 'install', '-q', *args]
    subprocess.run(command, check=True)

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[pdf-inspector]'],
    cwd=REPO_DIR,
    check=True,
)
print('Da cai dependency cua repository.')


In [ ]:
# Noi mapping da upload tren Drive; symlink de khong copy ~1.2 GB physics vao /content.
def link_vidore_page_map(subset: str) -> None:
    source = DRIVE_VIDORE_ROOT / f'vidore_v3_{subset}' / 'corpus'
    target = REPO_DIR / 'data' / 'raw' / f'vidore_v3_{subset}' / 'corpus'
    if not source.is_dir():
        raise FileNotFoundError(
            f'Thiếu mapping trên Drive: {source}. '
            'Hãy upload đủ thư mục corpus trước.'
        )
    source_files = sorted(source.glob('*.parquet'))
    if not source_files:
        raise FileNotFoundError(f'Không có parquet trong {source}')

    target.parent.mkdir(parents=True, exist_ok=True)
    if target.is_symlink():
        target.unlink()
    elif target.exists() and target.is_dir():
        existing = sorted(target.glob('*.parquet'))
        if existing:
            print(f'{subset}: reuse local mapping ({len(existing)} files)')
            return
        target.rmdir()
    try:
        target.symlink_to(source, target_is_directory=True)
        mode = 'symlink'
    except OSError:
        target.mkdir(parents=True, exist_ok=True)
        for path in source_files:
            shutil.copy2(path, target / path.name)
        mode = 'copy'
    print(f'{subset}: {len(source_files)} mapping files ({mode})')

for _subset in ('physics', 'industrial'):
    link_vidore_page_map(_subset)


In [ ]:
# PP-OCRv5 phai chay truoc vLLM trong cung Colab runtime.
pip_install(
    'paddlepaddle-gpu==3.0.0',
    '-i',
    'https://www.paddlepaddle.org.cn/packages/stable/cu126/',
)
pip_install('paddleocr>=3.0,<4')
pip_install(
    'nvidia-nccl-cu12==2.28.9',
    'nvidia-cudnn-cu12==9.19.0.56',
    'nvidia-cusparselt-cu12==0.7.1',
)

import paddle
print('Paddle:', paddle.__version__)
print('Paddle CUDA:', paddle.is_compiled_with_cuda())
if not paddle.is_compiled_with_cuda():
    raise RuntimeError('PaddlePaddle khong co CUDA support.')


In [ ]:
# Tao bundle chi tu pdf-inspector. Khong dung Tesseract de chon trang yeu.
if REBUILD_LIGHT_BUNDLE or not PP_OCR_BUNDLE.is_file():
    if not DRIVE_DOCBENCH_ROOT.is_dir():
        raise FileNotFoundError(f'Khong tim thay DocBench: {DRIVE_DOCBENCH_ROOT}')
    print('DocBench entries:', [path.name for path in sorted(DRIVE_DOCBENCH_ROOT.iterdir())[:20]])
    build_result = subprocess.run(
        [
            sys.executable,
            str(REPO_DIR / 'scripts' / 'build_ppocrv5_bundle.py'),
            '--docbench-root',
            str(DRIVE_DOCBENCH_ROOT),
            '--output',
            str(PP_OCR_BUNDLE),
        ],
        cwd=REPO_DIR,
        text=True,
        capture_output=True,
    )
    if build_result.stdout:
        print(build_result.stdout, end='')
    if build_result.stderr:
        print(build_result.stderr, end='', file=sys.stderr)
    if build_result.returncode != 0:
        raise RuntimeError(
            f'build_ppocrv5_bundle.py failed with exit code {build_result.returncode}'
        )
else:
    print('Reuse bundle:', PP_OCR_BUNDLE)

print('Bundle size MiB:', PP_OCR_BUNDLE.stat().st_size / 1024**2)


In [ ]:
# Giai nen va render tung trang PDF thanh PNG cho PP-OCRv5.
import fitz

OCR_EXTRACT_DIR = Path('/content/ppocrv5_light_ocr')
OCR_PNG_DIR = OCR_EXTRACT_DIR / 'png'
OCR_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
OCR_PNG_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(PP_OCR_BUNDLE) as archive:
    archive.extractall(OCR_EXTRACT_DIR)

pages = [
    json.loads(line)
    for line in (OCR_EXTRACT_DIR / 'pages.jsonl').read_text(encoding='utf-8').splitlines()
    if line.strip()
]
if not pages:
    raise RuntimeError('Bundle khong co trang can PP-OCRv5.')
# Protect against a stale/append-duplicated bundle manifest.
unique_pages = {}
duplicate_manifest_units = []
for page in pages:
    unit = str(page.get('unit') or page.get('page_id') or '')
    if not unit:
        raise RuntimeError('PP-OCR bundle contains a page without unit/page_id.')
    if unit in unique_pages:
        duplicate_manifest_units.append(unit)
    unique_pages[unit] = page
pages = list(unique_pages.values())
if duplicate_manifest_units:
    print(f'Deduplicated bundle manifest: {len(duplicate_manifest_units)} duplicate pages')

for index, page in enumerate(pages):
    source_pdf = OCR_EXTRACT_DIR / page['file']
    token = hashlib.sha1(str(page['unit']).encode('utf-8')).hexdigest()[:12]
    png_path = OCR_PNG_DIR / f'{index:05d}_{token}.png'
    if not png_path.is_file():
        with fitz.open(str(source_pdf)) as document:
            source_page = document[0]
            dpi = 200
            long_side = max(source_page.rect.width, source_page.rect.height) * dpi / 72
            if long_side > 4000:
                dpi = int(dpi * 4000 / long_side)
            source_page.get_pixmap(dpi=dpi, alpha=False).save(str(png_path))
    page['_png_path'] = str(png_path)

print(f'PP-OCRv5 candidates: {len(pages)}')
print(f'PNG directory: {OCR_PNG_DIR}')


In [ ]:
# Chay PP-OCRv5 theo page, ghi append de co the resume sau khi Colab bi ngat.

if RERUN_PP_OCR:
    PP_OCR_JSONL.write_text('', encoding='utf-8')

def read_latest_rows(path: Path) -> dict[str, dict]:
    latest = {}
    if not path.is_file():
        return latest
    for line in path.read_text(encoding='utf-8').splitlines():
        if not line.strip():
            continue
        try:
            row = json.loads(line)
        except json.JSONDecodeError:
            continue
        key = row.get('unit') or row.get('page_id') or row.get('id')
        if key:
            latest[str(key)] = row
    return latest

done_rows = {
    unit: row
    for unit, row in read_latest_rows(PP_OCR_JSONL).items()
    if not row.get('error')
}
pending = [page for page in pages if str(page['unit']) not in done_rows]
print(f'Existing successful OCR rows: {len(done_rows)}; pending: {len(pending)}')

ocr = None
if pending:
    from paddleocr import PaddleOCR
    ocr = PaddleOCR(
        text_detection_model_name='PP-OCRv5_server_det',
        text_recognition_model_name='PP-OCRv5_server_rec',
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
        device='gpu:0',
    )

def field(result, name, default=None):
    try:
        if hasattr(result, 'get'):
            return result.get(name, default)
        return result[name]
    except Exception:
        return getattr(result, name, default)

def run_one(page: dict) -> dict:
    started = time.perf_counter()
    try:
        texts = []
        scores = []
        for result in ocr.predict(page['_png_path']):
            values = field(result, 'rec_texts', [])
            confidences = field(result, 'rec_scores', [])
            values = list(values) if values is not None else []
            confidences = list(confidences) if confidences is not None else []
            texts.extend(str(value).strip() for value in values if str(value).strip())
            scores.extend(float(value) for value in confidences)
        text = '\n'.join(texts)
        return {
            'unit': page['unit'],
            'page_id': page.get('page_id', page['unit']),
            'doc_id': page.get('doc_id'),
            'page_index': page.get('page_index'),
            'page_number': page.get('page_number'),
            'text': text,
            'ocr_word_count': len(re.findall(r'\w+', text, flags=re.UNICODE)),
            'ocr_mean_confidence': statistics.mean(scores) if scores else None,
            'seconds': time.perf_counter() - started,
            'error': None,
        }
    except Exception as exc:
        return {
            'unit': page['unit'],
            'page_id': page.get('page_id', page['unit']),
            'doc_id': page.get('doc_id'),
            'page_index': page.get('page_index'),
            'page_number': page.get('page_number'),
            'text': '',
            'ocr_word_count': 0,
            'ocr_mean_confidence': None,
            'seconds': time.perf_counter() - started,
            'error': f'{type(exc).__name__}: {exc}'[:500],
        }

PP_OCR_JSONL.parent.mkdir(parents=True, exist_ok=True)
with PP_OCR_JSONL.open('a', encoding='utf-8') as output:
    for number, page in enumerate(pending, start=1):
        row = run_one(page)
        output.write(json.dumps(row, ensure_ascii=False) + '\n')
        output.flush()
        if number % 25 == 0 or number == len(pending):
            print(f'PP-OCRv5 {number}/{len(pending)}: {page["unit"]}')

final_rows = read_latest_rows(PP_OCR_JSONL)
missing = {str(page['unit']) for page in pages} - set(final_rows)
failures = [row for row in final_rows.values() if row.get('error')]
print(f'PP-OCRv5 rows: {len(final_rows)}; missing: {len(missing)}; failures: {len(failures)}')
if missing:
    raise RuntimeError(f'PP-OCRv5 con thieu {len(missing)} page rows.')

# Resume mode appends rows; compact to one latest row per current page so
# the strict page-index merge cannot see stale duplicate records.
ordered_rows = [final_rows[str(page['unit'])] for page in pages]
compact_path = PP_OCR_JSONL.with_name(PP_OCR_JSONL.name + '.compact.tmp')
with compact_path.open('w', encoding='utf-8') as output:
    for row in ordered_rows:
        output.write(json.dumps(row, ensure_ascii=False) + '\n')
compact_path.replace(PP_OCR_JSONL)
final_rows = {str(row['unit']): row for row in ordered_rows}
print(f'Compacted PP-OCRv5 artifact: {len(final_rows)} unique current pages')

# Phai giai phong Paddle truoc khi cài va chay vLLM tren L4.
if ocr is not None:
    del ocr
    gc.collect()
    try:
        import paddle
        paddle.device.cuda.empty_cache()
    except Exception as exc:
        print('Khong the empty Paddle CUDA cache:', exc)


In [ ]:
# Dong goi artifact OCR va chi sau do moi cai vLLM.
with zipfile.ZipFile(PP_OCR_RESULTS_ZIP, 'w', zipfile.ZIP_DEFLATED) as archive:
    archive.write(PP_OCR_JSONL, arcname='light_ocr_ppocrv5.jsonl')
    archive.writestr(
        'meta_colab.json',
        json.dumps(
            {
                'model': 'PP-OCRv5_server_det + PP-OCRv5_server_rec',
                'device': 'gpu:0',
                'source_bundle': str(PP_OCR_BUNDLE),
                'docbench_root': str(DRIVE_DOCBENCH_ROOT),
            },
            ensure_ascii=False,
            indent=2,
        ),
    )
print('PP-OCR JSONL:', PP_OCR_JSONL)
print('PP-OCR ZIP:', PP_OCR_RESULTS_ZIP)

# PaddleOCR installs a PaddleX vLLM plugin that KDL does not use.
# Remove both distributions before starting vLLM, matching the clean
# environment used by KDL_serving_de (2) (1).ipynb.
cleanup = subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'paddleocr', 'paddlex'],
    text=True,
    capture_output=True,
)
if cleanup.stdout:
    print(cleanup.stdout, end='')
if cleanup.stderr:
    print(cleanup.stderr, end='', file=sys.stderr)
if cleanup.returncode != 0:
    raise RuntimeError('Khong the go PaddleOCR/PaddleX truoc khi chay vLLM.')

# Fail early instead of starting a server that will load the stale plugin.
plugin_probe = subprocess.run(
    [
        sys.executable, '-c',
        (
            "from importlib.metadata import entry_points; "
            "eps = entry_points(group='vllm.general_plugins'); "
            "print('\\n'.join(ep.name + ' [' + (ep.dist.name if ep.dist else 'unknown') + ']' for ep in eps))"
        ),
    ],
    text=True,
    capture_output=True,
)
if plugin_probe.returncode != 0:
    raise RuntimeError(
        'Khong the kiem tra vLLM plugins: ' + plugin_probe.stderr.strip()
    )
remaining_plugins = plugin_probe.stdout.strip()
if remaining_plugins:
    raise RuntimeError(
        'Van con vLLM plugin sau khi go PaddleX; hay restart Colab runtime.\n'
        + remaining_plugins
    )
print('Da go PaddleOCR/PaddleX; khong con vLLM general plugin.')

pip_install('vllm==0.19.0')
print('Da cai vLLM.')


In [ ]:
# Khoi dong KDL/vLLM local theo cau hinh L4 cua notebook KDL serving.
import torch

gpu = torch.cuda.get_device_properties(0)
if 'L4' not in gpu.name:
    raise RuntimeError(f'KDL cell dang chay tren GPU khac L4: {gpu.name}')

VLLM_API_BASE = 'http://127.0.0.1:8000/v1'
VLLM_MODEL_NAME = 'kdl-frontier-parser-nano'
VLLM_DTYPE = 'bfloat16'
VLLM_MAX_NUM_SEQS = 64
VLLM_MAX_BATCHED_TOKENS = 8192
VLLM_GPU_MEMORY_UTILIZATION = 0.90
KDL_MAX_WORKERS = 256
KDL_BBOX_MAX_WORKERS = 128
KDL_RENDER_PROCESSES = min(32, os.cpu_count() or 1)
KDL_REQUEST_WORKERS = 64
KDL_MAX_MODEL_SEQUENCES = 64
KDL_REQUEST_BATCH_SIZE = 1

VLLM_LOG_PATH = RUN_OUTPUT_DIR / 'vllm.log'
VLLM_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
PID_PATH = Path('/content/vllm_kdl.pid')
STARTED_VLLM_PID = None

def process_is_vllm(pid: int) -> bool:
    try:
        args = subprocess.check_output(
            ['ps', '-p', str(pid), '-o', 'args='], text=True
        ).strip()
        return 'vllm' in args and 'serve' in args
    except Exception:
        return False

existing_pid = None
if PID_PATH.is_file():
    try:
        candidate = int(PID_PATH.read_text().strip())
        if process_is_vllm(candidate):
            existing_pid = candidate
    except (ValueError, OSError):
        pass

if existing_pid is None:
    vllm_binary = shutil.which('vllm')
    if not vllm_binary:
        raise RuntimeError('Khong tim thay executable vllm.')
    command = [
        vllm_binary, 'serve', 'KDLAI/KDL-Frontier-Parser-nano',
        '--host', '127.0.0.1',
        '--port', '8000',
        '--served-model-name', VLLM_MODEL_NAME,
        '--dtype', VLLM_DTYPE,
        '--max-model-len', '8192',
        '--max-num-seqs', str(VLLM_MAX_NUM_SEQS),
        '--max-num-batched-tokens', str(VLLM_MAX_BATCHED_TOKENS),
        '--gpu-memory-utilization', str(VLLM_GPU_MEMORY_UTILIZATION),
        '--limit-mm-per-prompt', '{"image":1}',
        '--trust-remote-code',
        '--enable-chunked-prefill',
        '--mm-processor-cache-gb', '0',
        '--generation-config', 'vllm',
    ]
    if VLLM_API_KEY:
        command += ['--api-key', VLLM_API_KEY]
    vllm_env = os.environ.copy()
    vllm_env['PYTHONUNBUFFERED'] = '1'
    vllm_env.pop('PYTHONPATH', None)
    # PaddleOCR/Colab may expose TensorFlow to Transformers; KDL does not need it.
    vllm_env['TRANSFORMERS_NO_TF'] = '1'
    vllm_env['USE_TF'] = '0'
    vllm_env['TF_CPP_MIN_LOG_LEVEL'] = '3'
    if HF_TOKEN:
        vllm_env['HF_TOKEN'] = HF_TOKEN
    log_handle = VLLM_LOG_PATH.open('a', encoding='utf-8', buffering=1)
    vllm_process = subprocess.Popen(
        command,
        cwd=REPO_DIR,
        env=vllm_env,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    log_handle.close()
    PID_PATH.write_text(str(vllm_process.pid), encoding='utf-8')
    STARTED_VLLM_PID = vllm_process.pid
    safe = [('<hidden>' if item == VLLM_API_KEY and VLLM_API_KEY else item) for item in command]
    print('Started:', ' '.join(shlex.quote(str(item)) for item in safe))
    print('PID:', STARTED_VLLM_PID)
else:
    STARTED_VLLM_PID = existing_pid
    print('Reuse running vLLM PID:', existing_pid)

print('vLLM log:', VLLM_LOG_PATH)


In [ ]:
# Cho endpoint KDL san sang.
import requests

os.environ['VLLM_API_BASE'] = VLLM_API_BASE
os.environ['VLLM_MODEL_NAME'] = VLLM_MODEL_NAME
if VLLM_API_KEY:
    os.environ['VLLM_API_KEY'] = VLLM_API_KEY
else:
    os.environ.pop('VLLM_API_KEY', None)

headers = {'Authorization': f'Bearer {VLLM_API_KEY}'} if VLLM_API_KEY else {}
deadline = time.time() + 30 * 60
models_payload = None
while time.time() < deadline:
    try:
        response = requests.get(f'{VLLM_API_BASE}/models', headers=headers, timeout=10)
        if response.status_code == 200:
            models_payload = response.json()
            break
        print('vLLM status:', response.status_code)
    except requests.RequestException:
        pass
    time.sleep(10)

if models_payload is None:
    print(VLLM_LOG_PATH.read_text(encoding='utf-8', errors='replace')[-20000:])
    raise RuntimeError('vLLM chua san sang sau 30 phut.')
models = [item.get('id') for item in models_payload.get('data', [])]
if VLLM_MODEL_NAME not in models:
    raise RuntimeError(f'Model KDL khong co trong endpoint: {models}')
print('KDL ready:', models_payload)


In [ ]:
# Tao command full run: retrieval + KDL parse + chunk/embed + QA + judge.
if not DRIVE_DOCBENCH_ROOT.is_dir():
    raise FileNotFoundError(f'Khong tim thay DocBench: {DRIVE_DOCBENCH_ROOT}')
CONFIG_PATH = REPO_DIR / CONFIG_REL
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f'Khong tim thay config: {CONFIG_PATH}')
if not PP_OCR_JSONL.is_file():
    raise FileNotFoundError(f'Khong tim thay PP-OCR artifact: {PP_OCR_JSONL}')

if OPENROUTER_API_KEY:
    os.environ['OPENROUTER_API_KEY'] = OPENROUTER_API_KEY
elif not SKIP_QA:
    raise RuntimeError('OPENROUTER_API_KEY la bat buoc khi chay QA.')

command = [
    sys.executable,
    '-m',
    'research.data_discovery.run_docbench_e2e',
    '--config', str(CONFIG_PATH),
    '--docbench-root', str(DRIVE_DOCBENCH_ROOT),
    '--retrieval-scope', RETRIEVAL_SCOPE,
    '--top-k-pages', str(TOP_K_PAGES),
    '--max-context-chunks', str(MAX_CONTEXT_CHUNKS),
    '--ppocr-jsonl', str(PP_OCR_JSONL),
    '--output-dir', str(RUN_OUTPUT_DIR),
    '--kdl-max-workers', str(KDL_MAX_WORKERS),
    '--kdl-render-processes', str(KDL_RENDER_PROCESSES),
    '--kdl-bbox-max-workers', str(KDL_BBOX_MAX_WORKERS),
    '--kdl-request-workers', str(KDL_REQUEST_WORKERS),
    '--kdl-request-batch-size', str(KDL_REQUEST_BATCH_SIZE),
    '--kdl-max-model-sequences', str(KDL_MAX_MODEL_SEQUENCES),
    '--kdl-host-failure-threshold', '20',
    '--query-workers', '32',
    '--qa-workers', '4',
    '--kdl-microbatch-window-seconds', '0.30',
    '--kdl-microbatch-max-pages', '512',
    '--log-level', 'INFO',
]
if MAX_DOCUMENTS is not None:
    command += ['--max-documents', str(MAX_DOCUMENTS)]
if QUESTION_LIMIT is not None:
    command += ['--limit', str(QUESTION_LIMIT)]
if FORCE_REPARSE:
    command += ['--force-reparse']
if FORCE_REBUILD_INDEX:
    command += ['--force-rebuild-index']
if SKIP_QA:
    command += ['--skip-qa']
if SKIP_JUDGE:
    command += ['--skip-judge']

safe_command = ' '.join(
    '<hidden>' if item == OPENROUTER_API_KEY and OPENROUTER_API_KEY else shlex.quote(str(item))
    for item in command
)
print(safe_command)
(RUN_OUTPUT_DIR / 'colab_run_config.json').write_text(
    json.dumps(
        {
            'repo_url': REPO_URL,
            'branch': GIT_BRANCH,
            'commit': commit,
            'docbench_root': str(DRIVE_DOCBENCH_ROOT),
            'output_dir': str(RUN_OUTPUT_DIR),
            'ppocr_jsonl': str(PP_OCR_JSONL),
            'command': [str(item) for item in command],
            'vllm_api_base': VLLM_API_BASE,
            'vllm_model': VLLM_MODEL_NAME,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding='utf-8',
)


In [ ]:
# Chay e2e va ghi mot console log rieng tren Drive.
expected_config = (REPO_DIR / CONFIG_REL).resolve()
if CONFIG_PATH.resolve() != expected_config:
    raise RuntimeError(
        f'CONFIG_PATH cu: {CONFIG_PATH}; hay chay lai cell Tao command full run.'
    )
try:
    command_config = Path(command[command.index('--config') + 1]).resolve()
except (ValueError, IndexError, NameError) as exc:
    raise RuntimeError('Chua tao command moi; hay chay lai cell Tao command full run.') from exc
if command_config != expected_config:
    raise RuntimeError(
        f'Command dang dung config cu: {command_config}; expected: {expected_config}. '
        'Hay chay lai cell Tao command full run.'
    )
print('Using config:', expected_config)
console_log = RUN_OUTPUT_DIR / 'logs' / 'colab_console.log'
console_log.parent.mkdir(parents=True, exist_ok=True)
run_env = os.environ.copy()
run_env['PYTHONUNBUFFERED'] = '1'
print('Bat dau full run. Log:', console_log)

with console_log.open('a', encoding='utf-8') as saved_output:
    saved_output.write(f'\nCOMMAND: {safe_command}\n')
    process = subprocess.Popen(
        command,
        cwd=REPO_DIR,
        env=run_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        saved_output.write(line)
        saved_output.flush()
    return_code = process.wait()

print('Runner exit code:', return_code)
if return_code != 0:
    raise RuntimeError(
        f'Pipeline that bai, exit code={return_code}. Xem logs trong {RUN_OUTPUT_DIR / "logs"}.'
    )


In [ ]:
# Doc report/timing va dong goi toan bo ket qua.
reports_dir = RUN_OUTPUT_DIR / 'reports'
report_files = sorted(reports_dir.glob(f'{RETRIEVAL_SCOPE}_baseline_legacy_ver*.json'))
timing_files = sorted(reports_dir.glob(f'{RETRIEVAL_SCOPE}_timing_summary_ver*.json'))
if report_files:
    report = json.loads(report_files[-1].read_text(encoding='utf-8'))
    print('Report:', report_files[-1])
    for key in (
        'questions_expected', 'questions_completed', 'accuracy',
        'correct_only', 'correct_plus_partial',
        'light_page_recall_at_10', 'light_page_recall_at_20',
        'light_page_ndcg_at_10', 'light_page_ndcg_at_20',
        'accurate_page_recall_at_10', 'accurate_page_recall_at_20',
        'bm25_gold_document_hit_rate',
    ):
        print(f'{key}: {report.get(key)}')
else:
    print('Chua co report; co the dang chay retrieval/QA hoac run da bi dung som.')

if timing_files:
    timing = json.loads(timing_files[-1].read_text(encoding='utf-8'))
    print('Timing:', timing_files[-1])
    for key in (
        'total_runtime_seconds_all_data',
        'phase_work_seconds_all_data',
        'online_latency_seconds_per_query',
        'infer_time_seconds_per_query',
        'light_preparation_seconds_all_data',
    ):
        print(f'{key}: {timing.get(key)}')

archive_base = RUN_OUTPUT_DIR.parent / RUN_NAME
archive_path = Path(shutil.make_archive(
    str(archive_base), 'zip', root_dir=RUN_OUTPUT_DIR.parent, base_dir=RUN_NAME
))
print('Complete result directory:', RUN_OUTPUT_DIR)
print('Complete result ZIP:', archive_path)
print('PP-OCR artifact ZIP:', PP_OCR_RESULTS_ZIP)


In [ ]:
# Cell tuy chon: chi dung vLLM neu notebook nay da khoi dong process.
if STOP_VLLM_AT_END and STARTED_VLLM_PID and process_is_vllm(STARTED_VLLM_PID):
    os.kill(STARTED_VLLM_PID, signal.SIGTERM)
    print('Da gui SIGTERM toi vLLM PID', STARTED_VLLM_PID)
else:
    print('Giữ vLLM dang chay; co the dung runtime Colab khi da lay xong ket qua.')
